# Cotton Detection Using Deep Learning (Google Colab Version)

This notebook is prepared for **Google Colab** so users can run the cotton detection workflow online without creating a local Conda or virtual environment.

This notebook will:

1. Install the required libraries in Colab
2. Help users upload or access the model and test image
3. Run cotton detection inference
4. Export the prediction as **GeoTIFF** and **GeoJSON**
5. Visualize the result in an **interactive map**

## Files needed

- `final_cotton.h5`
- `test_image.tif`

## Install Required Libraries

Google Colab already includes many standard Python packages, but geospatial libraries such as **GDAL**, **GeoPandas**, and related dependencies usually need to be installed manually.

Run the next cell first.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y gdal-bin libgdal-dev > /dev/null
!pip -q install geopandas folium shapely pyproj fiona rasterio
!pip -q install GDAL==$(gdal-config --version)

## Add the Model and Test Image

You need these two files before running the notebook:

- `final_cotton.h5`
- `test_image.tif`

### Easy ways to use the files in Colab

**Option 1: Keep the files on your computer**
- Download the model and test image from the GitHub repository to your computer.
- Run the upload cell below.
- Select both files from your computer when Colab opens the file picker.

**Option 2: Keep the files in Google Drive**
- Upload both files to a folder in your Google Drive.
- Mount Google Drive in Colab using the optional section below.
- Update the file paths in the notebook if you want to load them directly from Drive.

**Tip:**  
For small demo files, direct upload from your computer is easier.  
For larger files, Google Drive is usually more convenient.

In [ ]:
#from google.colab import files

#uploaded = files.upload()

## Optional: Use Google Drive

Use this only if you want to read the model and test image directly from Google Drive instead of uploading them every time.


In [ ]:
 #from google.colab import drive
 #drive.mount('/content/drive')

# Example:
 model_path = '/content/final_cotton.h5'
 image_path = '/content/test_image.tif'

## 1. Import Required Libraries

Libraries used:

- NumPy → numerical processing
- TensorFlow/Keras → deep learning inference
- GDAL → raster processing
- OGR → vector processing
- GeoPandas → vector analysis
- Folium → interactive maps

References:
https://numpy.org  
https://tensorflow.org  
https://gdal.org  
https://geopandas.org  
https://python-visualization.github.io/folium/

In [ ]:
!pip install tensorflow
import os
import random
import io
import base64

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Conv2DTranspose
from tensorflow.keras import backend as K
from tensorflow.keras.losses import binary_crossentropy

from osgeo import gdal, ogr, osr

import geopandas as gpd
import folium

from PIL import Image
from shapely.geometry import Polygon

## 2. Utility Functions

Satellite images contain multiple spectral bands. These bands often have different numeric ranges.

To stabilize model predictions we normalize each band between 0 and 1.

Feature scaling explanation:
https://en.wikipedia.org/wiki/Feature_scaling

In [ ]:

def scale_channels(data):

    scaled_data = np.zeros_like(data, dtype=np.float32)

    for i in range(data.shape[-1]):
        ch_min = np.min(data[..., i])
        ch_max = np.max(data[..., i])

        if ch_max - ch_min != 0:
            scaled_data[..., i] = (data[..., i] - ch_min) / (ch_max - ch_min)
        else:
            scaled_data[..., i] = 0.0

    return scaled_data

## 3. Custom Loss Functions

The model was trained using Dice Loss combined with Binary Cross Entropy.

Dice coefficient measures the overlap between predicted and ground truth masks.

https://en.wikipedia.org/wiki/Sørensen–Dice_coefficient

In [ ]:
def dice_loss(y_true, y_pred):
    smooth = 1.0

    y_true_f = K.flatten(tf.cast(y_true, tf.float32))
    y_pred_f = K.flatten(y_pred)

    intersection = K.sum(y_true_f * y_pred_f)

    return 1 - (2. * intersection + smooth) / (
        K.sum(y_true_f) + K.sum(y_pred_f) + smooth
    )

def dice_bce_loss(y_true, y_pred):
    return binary_crossentropy(y_true, y_pred) + dice_loss(y_true, y_pred)

def f1_score(y_true, y_pred):

    y_pred = tf.cast(y_pred > 0.5, tf.float32)

    tp = tf.reduce_sum(y_true * y_pred)
    fp = tf.reduce_sum((1 - y_true) * y_pred)
    fn = tf.reduce_sum(y_true * (1 - y_pred))

    return 2 * tp / (2 * tp + fp + fn + K.epsilon())

## 4. Load Pretrained Model

Load the trained cotton detection segmentation model.

In [ ]:
# 1. Define a 'fix' for the Conv2DTranspose layer
class FixedConv2DTranspose(Conv2DTranspose):
    @classmethod
    def from_config(cls, config):
        # If 'groups' is in the saved file, remove it before initializing
        config.pop('groups', None)
        return super().from_config(config)

# Define the custom loss functions as they were originally (functions, not classes)
def dice_loss(y_true, y_pred):
    smooth = 1.0
    y_true_f = K.flatten(tf.cast(y_true, tf.float32))
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (
        K.sum(y_true_f) + K.sum(y_pred_f) + smooth
    )

def dice_bce_loss(y_true, y_pred):
    return binary_crossentropy(y_true, y_pred) + dice_loss(y_true, y_pred)

# Define the custom metric function as it was originally (function, not class)
def f1_score(y_true, y_pred):
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    tp = tf.reduce_sum(y_true * y_pred)
    fp = tf.reduce_sum((1 - y_true) * y_pred)
    fn = tf.reduce_sum(y_true * (1 - y_pred))
    return 2 * tp / (2 * tp + fp + fn + K.epsilon())

# 2. Load the model with the fix included in custom_objects,
#    but disable immediate compilation to avoid the error.
model = keras.models.load_model(
    "final_cotton.h5",
    custom_objects={
        'dice_loss': dice_loss,
        'dice_bce_loss': dice_bce_loss,
        'f1_score': f1_score,
        'Conv2DTranspose': FixedConv2DTranspose
    },
    compile=False  # Crucial: Prevent Keras 3 from recompiling with legacy training config
)

# 3. Manually compile the model after loading its structure and weights
#    Assuming 'adam' optimizer was used and these are the intended loss and metrics.
model.compile(
    optimizer='adam',
    loss=dice_bce_loss,
    metrics=[f1_score]
)


## 5. Load Satellite Image

Read the GeoTIFF satellite image using GDAL.

In [ ]:
image_path = "test_image.tif"

img = gdal.Open(image_path)
if img is None:
    raise FileNotFoundError(f"Could not open image file: {image_path}")

transform = img.GetGeoTransform()
projection = img.GetProjection()
raster_count = img.RasterCount # Store RasterCount for later use

height = img.RasterYSize
width = img.RasterXSize

# Do NOT load the entire image into a NumPy array here to save memory.
# Patches will be read directly from 'img' in the prediction loop.

print(f"Image dimensions: Height={height}, Width={width}, Bands={raster_count}")

## 6. Generate Image Patches

Large rasters cannot be processed directly by deep learning models. Therefore we divide the raster into 256×256 patches using a sliding window.

In [ ]:
patch_size = 256
overlap = 0.5

# This list will now store only the coordinates of each patch, not the patch data itself
patch_coords = []

for x in range(0, width - 1, int(patch_size * (1 - overlap))):
    for y in range(0, height - 1, int(patch_size * (1 - overlap))):

        xmin = min(x, width - patch_size)
        ymin = min(y, height - patch_size)

        # Store only coordinates
        patch_coords.append((xmin, ymin))

print("Total patches (coordinates stored):", len(patch_coords))

## 7. Model Prediction

Each patch is normalized and passed to the neural network to predict cotton presence.

In [ ]:
best_threshold = 0.4677
batch_size = 32

empty_img = np.zeros((height, width, 1), dtype=np.float32)
overlay_count = np.zeros((height, width, 1), dtype=np.int32)

print("Starting prediction and image rebuilding...")

for i in range(0, len(patch_coords), batch_size):

    # Get the current batch of patch coordinates
    current_batch_coords = patch_coords[i : i + batch_size]

    # Extract patches on-the-fly for the current batch directly from the GDAL dataset
    batch_patches_data = []
    # Define band list for ReadAsArray (GDAL bands are 1-indexed)
    band_list_for_reading = list(range(1, raster_count + 1))

    for xmin, ymin in current_batch_coords:
        # Read a patch directly from the GDAL dataset 'img'
        # Output will be (bands, patch_size, patch_size)
        patch_array = img.ReadAsArray(xoff=xmin, yoff=ymin,
                                      xsize=patch_size, ysize=patch_size,
                                      band_list=band_list_for_reading)

        # Transpose to (patch_size, patch_size, bands) for model input
        # and ensure the data type is int32 as it was when the entire image was loaded.
        patch = patch_array.transpose(1, 2, 0).astype(np.int32)
        batch_patches_data.append(patch)

    # Convert list of patches to a numpy array for prediction
    batch_array = np.array(batch_patches_data)

    # Normalize and predict
    normalized_batch = scale_channels(batch_array)
    batch_pred = model.predict(normalized_batch, verbose=0)

    # Apply threshold to predictions
    batch_predictions_thresholded = (batch_pred > best_threshold).astype(np.uint8)

    # Rebuild the full image with predictions from the current batch
    for index_in_batch, (xmin, ymin) in enumerate(current_batch_coords):

        img_slice = batch_predictions_thresholded[index_in_batch]

        x1 = xmin + patch_size
        y1 = ymin + patch_size

        empty_img[ymin:y1, xmin:x1] += img_slice
        overlay_count[ymin:y1, xmin:x1] += 1

print("Prediction and initial rebuilding completed.")

# Final calculation after all batches are processed
# Handle potential division by zero for areas not covered by any patch
# (though with overlap, this should be rare for interior regions)
overlay_count_safe = np.where(overlay_count == 0, 1, overlay_count) # Avoid division by zero
rebuilt_img = empty_img / overlay_count_safe

binary = np.where(tf.squeeze(rebuilt_img) > best_threshold, 255, 0)
print("Final binary mask generated.")

## 7.1 Visualize Sample Patches and Predictions

Let's visualize a few random patches and their corresponding predictions to see how the model is performing.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Select a few random patches to visualize, prioritizing those with 30-40% cotton coverage
num_samples = 5

# Find indices of patches that contain cotton within the specified percentage range (30-40%)
patches_with_specific_cotton_percentage = []
for idx, (xmin, ymin) in enumerate(patch_coords):
    mask_patch = binary[ymin : ymin + patch_size, xmin : xmin + patch_size]
    total_pixels_in_patch = patch_size * patch_size
    cotton_pixels = np.sum(mask_patch == 255)
    cotton_percentage = (cotton_pixels / total_pixels_in_patch) * 100

    if 30 <= cotton_percentage <= 40:
        patches_with_specific_cotton_percentage.append(idx)

# Prioritize patches within the 30-40% range
if len(patches_with_specific_cotton_percentage) >= num_samples:
    random_indices = random.sample(patches_with_specific_cotton_percentage, num_samples)
else:
    # If not enough in the specific range, take all of them
    random_indices = patches_with_specific_cotton_percentage

    # Then, if still not enough, supplement with other patches that have *any* cotton (not just the 30-40% range)
    remaining_samples = num_samples - len(patches_with_specific_cotton_percentage)
    if remaining_samples > 0:
        patches_with_any_cotton_indices = []
        for idx, (xmin, ymin) in enumerate(patch_coords):
            mask_patch = binary[ymin : ymin + patch_size, xmin : xmin + patch_size]
            if np.any(mask_patch == 255) and idx not in random_indices: # Ensure no duplicates
                patches_with_any_cotton_indices.append(idx)

        if len(patches_with_any_cotton_indices) > 0:
            random_indices.extend(random.sample(patches_with_any_cotton_indices, min(remaining_samples, len(patches_with_any_cotton_indices))))

    # If still not enough, just use what we have, up to num_samples
    random_indices = random_indices[:num_samples]

# Create a custom colormap for the prediction mask: white for 0 (non-cotton), lime green for 255 (cotton)
colors = ['white', 'lime'] # 0 (non-cotton) maps to white, 255 (cotton) maps to lime green
cmap = mcolors.ListedColormap(colors)

# Increase figure size for better visualization
plt.figure(figsize=(25, 10))

for i, idx in enumerate(random_indices):
    if i >= num_samples: # Ensure we don't try to plot more than num_samples
        break

    xmin, ymin = patch_coords[idx]

    # Get the original patch from the image by reading directly from the GDAL dataset 'img'
    # Define band list for ReadAsArray (GDAL bands are 1-indexed)
    band_list_for_reading = list(range(1, raster_count + 1))
    patch_array_gdal = img.ReadAsArray(xoff=xmin, yoff=ymin,
                                   xsize=patch_size, ysize=patch_size,
                                   band_list=band_list_for_reading)
    original_patch = patch_array_gdal.transpose(1, 2, 0).astype(np.int32)

    # Normalize the first 3 bands (RGB) of the original_patch for display
    normalized_original_patch_rgb = scale_channels(original_patch[:, :, :3])

    # Get the predicted mask for this patch from the 'binary' array
    predicted_mask_patch = binary[ymin : ymin + patch_size, xmin : xmin + patch_size]

    # Display original patch (using the normalized first 3 bands for RGB)
    plt.subplot(2, num_samples, i + 1)
    plt.imshow(normalized_original_patch_rgb)
    plt.title(f"Original Patch {idx}")
    plt.axis("off")

    # Display predicted mask with custom colormap
    plt.subplot(2, num_samples, i + 1 + num_samples)
    plt.imshow(predicted_mask_patch, cmap=cmap, vmin=0, vmax=255) # Use vmin/vmax to map 0/255 to colors
    plt.title(f"Prediction {idx}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 8. Export Prediction Mask as GeoTIFF

In [ ]:
output_folder = "prediction_output"
os.makedirs(output_folder, exist_ok=True)

output_path = os.path.join(output_folder, "predicted_mask.tif")

driver = gdal.GetDriverByName("GTiff")

out_ds = driver.Create(
    output_path,
    width,
    height,
    1,
    gdal.GDT_Byte
)

out_ds.SetGeoTransform(transform)
out_ds.SetProjection(projection)

out_ds.GetRasterBand(1).WriteArray(binary)

out_ds.FlushCache()
out_ds = None

print("GeoTIFF saved:", output_path)

## 9. Convert Raster Mask to GeoJSON

In [ ]:
vector_path = os.path.join(output_folder, "cotton_patches.geojson")

driver = ogr.GetDriverByName("GeoJSON")
output_ds = driver.CreateDataSource(vector_path)

srs = osr.SpatialReference()
srs.ImportFromWkt(projection)

layer = output_ds.CreateLayer("cotton_patches", srs, ogr.wkbPolygon)

field = ogr.FieldDefn("value", ogr.OFTInteger)
layer.CreateField(field)

mem_driver = gdal.GetDriverByName('MEM')

mem_ds = mem_driver.Create('', width, height, 1, gdal.GDT_Byte)

mem_ds.SetGeoTransform(transform)
mem_ds.SetProjection(projection)

mem_ds.GetRasterBand(1).WriteArray(binary)

gdal.Polygonize(mem_ds.GetRasterBand(1), None, layer, 0)

output_ds = None
mem_ds = None

print("GeoJSON saved:", vector_path)

## 10. Interactive Map Visualization

This section visualizes the prediction results in an interactive web map using **Folium**.

The map will display:

• the satellite image  
• predicted cotton polygons  
• interactive tooltips  

Folium documentation:
https://python-visualization.github.io/folium/

In [ ]:
import os

geojso_filepath = os.path.join(output_folder, "cotton_patches.geojson")

gdf = gpd.read_file(geojso_filepath)

gdf["geometry"] = gdf.geometry.make_valid()
gdf["geometry"] = gdf.geometry.buffer(0)

gdf = gdf[gdf.geometry.notna()]
gdf = gdf[~gdf.geometry.is_empty]

gdf_wgs84 = gdf.to_crs(epsg=4326)

# Define scale_factor here to apply during warping
scale_factor = 2

# Calculate target dimensions for the warped image
target_width = img.RasterXSize // scale_factor
target_height = img.RasterYSize // scale_factor

# Warp and resample the image directly to a smaller resolution in one go
warped = gdal.Warp(
    "", # output to memory
    img,
    format="MEM",
    dstSRS="EPSG:4326",
    resampleAlg=gdal.GRA_Bilinear,
    width=target_width,   # Downsample during warping
    height=target_height  # Downsample during warping
)

nx = warped.RasterXSize # This will now be the target_width
ny = warped.RasterYSize # This will now be the target_height
gt = warped.GetGeoTransform() # This will be the geotransform of the resampled image

rgb = np.dstack([
    warped.GetRasterBand(1).ReadAsArray(),
    warped.GetRasterBand(2).ReadAsArray(),
    warped.GetRasterBand(3).ReadAsArray()
]).astype(np.float32)

# Close the warped dataset to free memory as soon as its data is read
warped = None # This will release the memory held by the warped GDAL dataset

rgb_min = rgb.min(axis=(0,1))
rgb_max = rgb.max(axis=(0,1))

rgb_norm = (rgb - rgb_min) / (rgb_max - rgb_min)

brightness_factor = 1.2
rgb_norm = np.clip(rgb_norm * brightness_factor, 0, 1)

pil_img = Image.fromarray((rgb_norm * 255).astype(np.uint8)) # Use rgb_norm directly as it's already scaled

buffer = io.BytesIO()
pil_img.save(buffer, format="PNG")

encoded_image = base64.b64encode(buffer.getvalue()).decode()

min_lon = gt[0]
max_lat = gt[3]

max_lon = gt[0] + gt[1] * nx
min_lat = gt[3] + gt[5] * ny

image_bounds = [[min_lat, min_lon], [max_lat, max_lon]]

image_extent_polygon = Polygon([
    (min_lon, min_lat),
    (max_lon, min_lat),
    (max_lon, max_lat),
    (min_lon, max_lat),
    (min_lon, min_lat)
])

gdf_filtered = gdf_wgs84[~gdf_wgs84.geometry.apply(lambda geom: geom.intersects(image_extent_polygon.boundary))]

center_lat = (min_lat + max_lat) / 2
center_lon = (min_lon + max_lon) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles=None, max_bounds=image_bounds)

folium.raster_layers.ImageOverlay(
    image=f"data:image/png;base64,{encoded_image}",
    bounds=image_bounds,
    opacity=0.7,
    name="Satellite Image"
).add_to(m)

folium.GeoJson(
    gdf_filtered.to_json(),
    name="Cotton Patches",
    style_function=lambda feature: {
        "fillColor": "#FF8888",
        "color": "#FF4444",
        "weight": 1,
        "fillOpacity": 0
    },
    tooltip=folium.GeoJsonTooltip(fields=["value"])
).add_to(m)

folium.LayerControl().add_to(m)

m.fit_bounds(image_bounds)

m